# Fine-tuning du modèle de segmentation de pyannote sur AMI

**But :** adapter le réseau de *segmentation* de `pyannote/speaker-diarization-community-1`
(celui qui dit « qui parle quand » dans chaque fenêtre de 10 s) à des réunions enregistrées avec un micro distant.

**Pourquoi la segmentation et pas les embeddings ?** Sur VoxConverse dev-10, l'erreur de pyannote vient surtout
de la parole ratée et des fausses alertes (DER strict : 2,24 + 2,05 contre 1,21 de confusion). C'est le travail de la segmentation.
En plus, ce réseau est petit (environ 1,5 M de paramètres) : il s'entraîne vite.

**Avant de lancer (panneau de droite, *Settings*) :**
- *Accelerator* : **GPU P100** ou **GPU T4 x2** (on n'utilise qu'un GPU).
- *Internet* : **On**.
- *Add-ons → Secrets* : `HF_TOKEN` coché pour ce notebook.

**Deux modes** (cellule « Réglages ») :
- `MODE = "test"` : 1 fichier de données, 1 époque, environ 20 min. Sert à vérifier que tout marche et à **mesurer la durée d'une époque**.
- `MODE = "nuit"` : toutes les données, jusqu'à 30 époques, plafonné à 10 h (Kaggle coupe à 12 h).
  Lancer avec **Save Version → Save & Run All (Commit)** : ça tourne sans navigateur ouvert.

In [ ]:
# Réglages : la seule cellule à modifier
MODE = "test"          # "test" puis "nuit"
CONFIG = "sdm"         # "sdm" = un seul micro distant (proche d'un PC posé sur une table), "ihm" = micros-casques
LR = 1e-4              # taux d'apprentissage : 10x plus petit que le défaut (1e-3) pour ne pas « casser » ce que le modèle sait déjà
MAX_HEURES = 10        # sécurité : Kaggle arrête une session à 12 h

NB_FICHIERS_TRAIN = 1 if MODE == "test" else 19   # le jeu train est découpé en 19 fichiers parquet (~7 réunions chacun)
NB_FICHIERS_VAL = 1 if MODE == "test" else 3
MAX_EPOQUES = 1 if MODE == "test" else 30
print(MODE, CONFIG, NB_FICHIERS_TRAIN, NB_FICHIERS_VAL, MAX_EPOQUES)

## 1. Installation
Même version de pyannote que sur le PC, pour que le modèle entraîné se recharge sans surprise.

Piège : Kaggle charge numpy **en mémoire** dès le démarrage. Si pip remplace numpy **sur le disque**,
Python mélange les deux versions et plante (`cannot import name '_slice'`). On interdit donc à pip de changer numpy,
et on vérifie que les deux versions sont les mêmes. Les lignes `ERROR: pip's dependency resolver...` concernent
des paquets préinstallés de Kaggle qu'on n'utilise pas : on peut les ignorer.

In [ ]:
import subprocess, sys

commande = [sys.executable, "-m", "pip", "install", "-q", "pyannote.audio==4.0.7"]

# Si Kaggle a déjà chargé numpy en mémoire, on interdit à pip de le changer sur le disque
deja_charge = "numpy" in sys.modules
print("numpy déjà chargé avant l'installation :", deja_charge)
if deja_charge:
    import numpy
    commande.append(f"numpy=={numpy.__version__}")

r = subprocess.run(commande, capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-3000:])
assert r.returncode == 0, "pip a échoué : voir le message ci-dessus"

# Vérification : un Python neuf lit le numpy du disque, il doit être le même que celui en mémoire
import numpy
sur_disque = subprocess.run([sys.executable, "-c", "import numpy; print(numpy.__version__)"],
                            capture_output=True, text=True).stdout.strip()
assert numpy.__version__ == sur_disque, (f"numpy {numpy.__version__} en mémoire mais {sur_disque} sur le disque : "
                                         "Run -> Restart & Clear Outputs, puis Run All")
print("numpy cohérent :", numpy.__version__)

import pyannote.audio
print("pyannote.audio", pyannote.audio.__version__)

!nvidia-smi --query-gpu=name,memory.total --format=csv
!ffmpeg -version | head -1

In [ ]:
import io, json, os, shutil, time
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
import soundfile as sf
import torch
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
RACINE = Path("/tmp/ami")          # données de travail (pas sauvegardées : on les retélécharge à chaque fois)
SORTIE = Path("/kaggle/working")   # ce qui est sauvegardé à la fin (modèle, courbes, résumé)
print("GPU :", torch.cuda.is_available(), "| torch", torch.__version__)
print("Espace libre sur /tmp : %.0f Go" % (shutil.disk_usage("/tmp").free / 1e9))

## 2. Données : AMI depuis Hugging Face
Source : `diarizers-community/ami` (licence CC BY 4.0). Mêmes partitions que l'AMI « full corpus » :
136 réunions en train, 18 en validation, 16 en test. **On n'utilise jamais le test ici.**

Chaque réunion est réécrite au format attendu par pyannote :
- `wav/<id>.wav` : l'audio (16 kHz mono) ;
- `rttm/<id>.rttm` : qui parle quand (la référence) ;
- `uem/<id>.uem` : la zone annotée (ici, tout le fichier) ;
- `lists/<partition>.txt` : la liste des identifiants.

In [ ]:
for dossier in ["wav", "rttm", "uem", "lists"]:
    (RACINE / dossier).mkdir(parents=True, exist_ok=True)


def convertir(partition_hf, partition, nb_fichiers, total_fichiers):
    uris, heures = [], 0.0
    for n in range(nb_fichiers):
        nom = f"{CONFIG}/{partition_hf}-{n:05d}-of-{total_fichiers:05d}.parquet"
        chemin = hf_hub_download("diarizers-community/ami", nom, repo_type="dataset",
                                 token=HF_TOKEN, local_dir="/tmp/hf")
        # Lecture réunion par réunion pour ne pas tout charger en mémoire
        for lot in pq.ParquetFile(chemin).iter_batches(batch_size=1):
            ligne = lot.to_pylist()[0]
            audio, sr = sf.read(io.BytesIO(ligne["audio"]["bytes"]), dtype="float32")
            if audio.ndim > 1:
                audio = audio.mean(axis=1)
            assert sr == 16000, f"fréquence inattendue : {sr}"
            # Identifiant de la réunion (ex. EN2001a), tiré du nom du fichier d'origine
            chemin_origine = ligne["audio"].get("path") or f"{partition}{n}_{len(uris)}"
            uri = Path(chemin_origine).name.split(".")[0]
            duree = len(audio) / sr

            sf.write(RACINE / "wav" / f"{uri}.wav", audio, sr, subtype="PCM_16")
            with open(RACINE / "rttm" / f"{uri}.rttm", "w") as f:
                for debut, fin, locuteur in zip(ligne["timestamps_start"], ligne["timestamps_end"], ligne["speakers"]):
                    f.write(f"SPEAKER {uri} 1 {debut:.3f} {fin - debut:.3f} <NA> <NA> {locuteur} <NA> <NA>\n")
            (RACINE / "uem" / f"{uri}.uem").write_text(f"{uri} 1 0.000 {duree:.3f}\n")
            uris.append(uri)
            heures += duree / 3600
        os.remove(chemin)  # libère le disque
        print(f"{nom} converti")
    (RACINE / "lists" / f"{partition}.txt").write_text("\n".join(uris) + "\n")
    print(f"{partition} : {len(uris)} réunions, {heures:.1f} h d'audio")
    return len(uris), heures


debut = time.time()
nb_train, h_train = convertir("train", "train", NB_FICHIERS_TRAIN, 19)
nb_val, h_val = convertir("validation", "development", NB_FICHIERS_VAL, 3)
print(f"Préparation des données : {(time.time() - debut) / 60:.1f} min")

In [ ]:
# Contrôle visuel : une réunion, ses locuteurs, et la cohérence des temps
uri = (RACINE / "lists" / "train.txt").read_text().split()[0]
lignes = (RACINE / "rttm" / f"{uri}.rttm").read_text().splitlines()
duree = sf.info(RACINE / "wav" / f"{uri}.wav").duration
fin_max = max(float(l.split()[3]) + float(l.split()[4]) for l in lignes)
print(uri, f"{duree:.0f} s,", len(lignes), "tours de parole,",
      len({l.split()[7] for l in lignes}), "locuteurs, dernier tour finit à", f"{fin_max:.0f} s")
assert fin_max <= duree + 1, "les temps du RTTM dépassent la durée de l'audio"

## 3. Protocole pyannote
Un petit fichier YAML dit à pyannote où trouver l'audio, les références et les listes.

In [ ]:
from pyannote.database import FileFinder, registry

yml = f'''
Databases:
  AMI: {RACINE}/wav/{{uri}}.wav

Protocols:
  AMI:
    SpeakerDiarization:
      {CONFIG}:
        train:
          uri: {RACINE}/lists/train.txt
          annotation: {RACINE}/rttm/{{uri}}.rttm
          annotated: {RACINE}/uem/{{uri}}.uem
        development:
          uri: {RACINE}/lists/development.txt
          annotation: {RACINE}/rttm/{{uri}}.rttm
          annotated: {RACINE}/uem/{{uri}}.uem
'''
(RACINE / "database.yml").write_text(yml)
registry.load_database(str(RACINE / "database.yml"))
protocole = registry.get_protocol(f"AMI.SpeakerDiarization.{CONFIG}", preprocessors={"audio": FileFinder()})
premier = next(protocole.train())
print(premier["uri"], premier["audio"], len(premier["annotation"].labels()), "locuteurs")

## 4. Modèle et tâche
- On part des poids de segmentation de `community-1` (pas d'un réseau vide) : c'est ça, le *fine-tuning*.
- La tâche reprend exactement les réglages du modèle d'origine : fenêtres de 10 s, au plus 3 locuteurs par fenêtre,
  au plus 2 en même temps. Sinon la dernière couche ne correspondrait plus.

In [ ]:
from pyannote.audio import Model
from pyannote.audio.tasks import SpeakerDiarization

modele = Model.from_pretrained("pyannote/speaker-diarization-community-1", subfolder="segmentation", token=HF_TOKEN)
specs = modele.specifications
print("durée", specs.duration, "| locuteurs par fenêtre", len(specs.classes), "| simultanés", specs.powerset_max_classes)

tache = SpeakerDiarization(
    protocole,
    duration=specs.duration,
    max_speakers_per_chunk=len(specs.classes),
    max_speakers_per_frame=specs.powerset_max_classes,
    batch_size=32,
    num_workers=4,       # Kaggle donne 4 cœurs CPU pour préparer les lots pendant que le GPU calcule
)
modele.task = tache


def configure_optimizers():
    return torch.optim.Adam(modele.parameters(), lr=LR)


modele.configure_optimizers = configure_optimizers  # remplace le taux par défaut (1e-3)

## 5. Score **avant** entraînement
On mesure le modèle d'origine sur la validation AMI. Sans ce point de départ, impossible de dire si le fine-tuning aide.

⚠️ Ce DER est calculé **par fenêtre de 10 s**, pas sur des fichiers entiers. Il sert à comparer avant/après ici,
pas à comparer avec les DER de `results/runs.csv`. Le vrai test se fera sur le PC avec `whospoke.evaluate`.

In [ ]:
from lightning.pytorch import Trainer

# inference_mode=False : sinon les compteurs du score sont figés et l'entraînement qui suit plante
avant = Trainer(accelerator="gpu", devices=1, logger=False, enable_checkpointing=False,
                inference_mode=False).validate(modele)[0]
nom_metrique, sens = tache.val_monitor
print("Métrique suivie :", nom_metrique, f"({sens})", "| avant :", round(avant[nom_metrique], 4))

## 6. Entraînement
- **ModelCheckpoint** garde le meilleur modèle sur la validation (et le dernier, au cas où la session coupe).
- **EarlyStopping** arrête si la validation ne s'améliore plus pendant 5 époques : inutile de continuer, et ça évite de sur-apprendre.
- **CSVLogger** écrit les scores de chaque époque, pour tracer la courbe.

In [ ]:
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger

sauvegarde = ModelCheckpoint(dirpath=str(SORTIE / "checkpoints"), monitor=nom_metrique, mode=sens,
                             save_top_k=1, save_last=True, filename="meilleur-{epoch:02d}")
arret = EarlyStopping(monitor=nom_metrique, mode=sens, patience=5)
journal = CSVLogger(str(SORTIE), name="logs")

entraineur = Trainer(accelerator="gpu", devices=1, max_epochs=MAX_EPOQUES, max_time={"hours": MAX_HEURES},
                     callbacks=[sauvegarde, arret], logger=journal)
debut = time.time()
entraineur.fit(modele)
duree_entrainement = time.time() - debut
print(f"Entraînement : {duree_entrainement / 60:.1f} min, {entraineur.current_epoch} époque(s)")
print(f"Soit {duree_entrainement / 60 / max(entraineur.current_epoch, 1):.1f} min par époque")

## 7. Résultats : courbe, meilleur modèle, résumé

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metriques = pd.read_csv(Path(journal.log_dir) / "metrics.csv")
val = metriques.dropna(subset=[nom_metrique])[["epoch", nom_metrique]]
print(val.to_string(index=False))

plt.figure(figsize=(6, 3.5))
plt.axhline(avant[nom_metrique], color="grey", linestyle="--", label="avant fine-tuning")
plt.plot(val["epoch"], val[nom_metrique], marker="o", label="après chaque époque")
plt.xlabel("époque"); plt.ylabel(nom_metrique + " (validation AMI, par fenêtre)")
plt.title(f"Fine-tuning segmentation sur AMI {CONFIG}"); plt.legend(); plt.tight_layout()
plt.savefig(SORTIE / "courbe_validation.png", dpi=150)
plt.show()

meilleur = Path(sauvegarde.best_model_path)
shutil.copy(meilleur, SORTIE / "segmentation_finetunee.ckpt")
resume = {
    "mode": MODE, "config": CONFIG, "lr": LR,
    "reunions_train": nb_train, "heures_train": round(h_train, 1),
    "reunions_val": nb_val, "heures_val": round(h_val, 1),
    "epoques": entraineur.current_epoch, "duree_min": round(duree_entrainement / 60, 1),
    "metrique": nom_metrique, "avant": round(avant[nom_metrique], 4),
    "meilleur": round(float(sauvegarde.best_model_score), 4), "fichier": meilleur.name,
}
(SORTIE / "resume.json").write_text(json.dumps(resume, indent=2, ensure_ascii=False))
print(json.dumps(resume, indent=2, ensure_ascii=False))